In [1]:
import pandas as pd
import re
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler

df = pd.read_csv('netflix_titles.csv')
df.head()

df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df.drop(columns='date_added', inplace=True)
df_clean = df.dropna()
df_clean_backup = df_clean.copy()

df_clean.to_csv('netflix_clean.csv', index=False, encoding='utf-8-sig')

In [ ]:
mlb = MultiLabelBinarizer()

df_clean = df_clean_backup.copy()

df_clean['cast'] = df_clean['cast'].apply(lambda x: [i.strip() for i in str(x).split(',')])

one_hot = pd.DataFrame(mlb.fit_transform(df_clean['cast']), columns=[f"cast_{cls}" for cls in mlb.classes_], dtype=int)

df_clean = df_clean.reset_index(drop=True)
one_hot = one_hot.reset_index(drop=True)

df_encoded = pd.concat([df_clean.drop(columns='cast'), one_hot], axis=1)

In [3]:
df_encoded['director'] = df_encoded['director'].apply(lambda x: [i.strip() for i in str(x).split(',')])

one_hot = pd.DataFrame(mlb.fit_transform(df_encoded['director']), columns=[f"director_{cls}" for cls in mlb.classes_], dtype=int)

df_encoded = df_encoded.reset_index(drop=True)
one_hot = one_hot.reset_index(drop=True)

df_encoded = pd.concat([df_encoded.drop(columns='director'), one_hot], axis=1)

In [4]:
df_encoded['country'] = df_encoded['country'].apply(lambda x: [i.strip() for i in str(x).split(',')])

one_hot = pd.DataFrame(mlb.fit_transform(df_encoded['country']), columns=[f"country_{cls}" for cls in mlb.classes_], dtype=int)

df_encoded = df_encoded.reset_index(drop=True)
one_hot = one_hot.reset_index(drop=True)

df_encoded = pd.concat([df_encoded.drop(columns='country'), one_hot], axis=1)

In [5]:
df_encoded['listed_in'] = df_encoded['listed_in'].apply(lambda x: [i.strip() for i in str(x).split(',')])

one_hot = pd.DataFrame(mlb.fit_transform(df_encoded['listed_in']), columns=[f"listed_in_{cls}" for cls in mlb.classes_], dtype=int)

df_encoded = df_encoded.reset_index(drop=True)
one_hot = one_hot.reset_index(drop=True)

df_encoded = pd.concat([df_encoded.drop(columns='listed_in'), one_hot], axis=1)

In [6]:
rating_adultness = {
    'TV-Y': 0,
    'TV-Y7': 1,
    'TV-Y7-FV': 2,
    'TV-G': 3,
    'G': 3,
    'TV-PG': 4,
    'PG': 4,
    'TV-14': 5,
    'PG-13': 6,
    'R': 7,
    'TV-MA': 8,
    'NC-17': 9,
    'NR': -1,
    'UR': -1
}
df_encoded['rating'] = df_encoded['rating'].map(rating_adultness)

In [7]:
numeric_sums = df_encoded.select_dtypes(include='number').sum()
numeric_cols_to_keep = numeric_sums[numeric_sums >= 5].index.tolist()

# Colonne non numeriche
non_numeric_cols = df_encoded.select_dtypes(exclude='number').columns.tolist()

# Unisci entrambe le liste
cols_to_keep =  non_numeric_cols + numeric_cols_to_keep

# Filtra il DataFrame
df_filtered = df_encoded[cols_to_keep]
df_filtered.head()

,show_id,type,title,duration,description,release_year,rating,cast_50 Cent,cast_Aamir Bashir,cast_Aamir Khan,...,listed_in_TV Action & Adventure,listed_in_TV Comedies,listed_in_TV Dramas,listed_in_TV Horror,listed_in_TV Mysteries,listed_in_TV Sci-Fi & Fantasy,listed_in_TV Shows,listed_in_TV Thrillers,listed_in_Teen TV Shows,listed_in_Thrillers
0,s1,Movie,Dick Johnson Is Dead,90 min,"As her father nears the end of his life, filmm...",2020,6,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,s2,TV Show,Blood & Water,2 Seasons,"After crossing paths at a party, a Cape Town t...",2021,8,0,0,0,...,0,0,1,0,1,0,0,0,0,0
2,s3,TV Show,Ganglands,1 Season,To protect his family from a powerful drug lor...,2021,8,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,s4,TV Show,Jailbirds New Orleans,1 Season,"Feuds, flirtations and toilet talk go down amo...",2021,8,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,s5,TV Show,Kota Factory,2 Seasons,In a city of coaching centers known to train I...,2021,8,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [8]:
def extract_duration(value):
    match = re.search(r'\d+', str(value))
    return int(match.group()) if match else None

df_filtered['duration_num'] = df_filtered['duration'].apply(extract_duration)

df_filtered['duration_final'] = df_filtered.apply(
    lambda row: row['duration_num'] * 1 if row['type'] == 'Movie'
    else row['duration_num'] * -40 * 8 if row['type'] == 'TV Show'
    else None,
    axis=1
)

df_filtered = df_filtered.drop(['duration', 'type', 'duration_num'], axis=1)

df_filtered.head()

C:\Users\gabba\AppData\Local\Temp\ipykernel_15680\340751974.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['duration_num'] = df_filtered['duration'].apply(extract_duration)
C:\Users\gabba\AppData\Local\Temp\ipykernel_15680\340751974.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['duration_final'] = df_filtered.apply(


,show_id,title,description,release_year,rating,cast_50 Cent,cast_Aamir Bashir,cast_Aamir Khan,cast_Aaron Eckhart,cast_Aaron Paul,...,listed_in_TV Comedies,listed_in_TV Dramas,listed_in_TV Horror,listed_in_TV Mysteries,listed_in_TV Sci-Fi & Fantasy,listed_in_TV Shows,listed_in_TV Thrillers,listed_in_Teen TV Shows,listed_in_Thrillers,duration_final
0,s1,Dick Johnson Is Dead,"As her father nears the end of his life, filmm...",2020,6,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,90
1,s2,Blood & Water,"After crossing paths at a party, a Cape Town t...",2021,8,0,0,0,0,0,...,0,1,0,1,0,0,0,0,0,-640
2,s3,Ganglands,To protect his family from a powerful drug lor...,2021,8,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,-320
3,s4,Jailbirds New Orleans,"Feuds, flirtations and toilet talk go down amo...",2021,8,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,-320
4,s5,Kota Factory,In a city of coaching centers known to train I...,2021,8,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,-640


In [10]:
numeric_cols = df_filtered.select_dtypes(include='number').columns

scaler = MinMaxScaler()
df_filtered[numeric_cols] = scaler.fit_transform(df_filtered[numeric_cols])

df_filtered.head()

,show_id,title,description,release_year,rating,cast_50 Cent,cast_Aamir Bashir,cast_Aamir Khan,cast_Aaron Eckhart,cast_Aaron Paul,...,listed_in_TV Comedies,listed_in_TV Dramas,listed_in_TV Horror,listed_in_TV Mysteries,listed_in_TV Sci-Fi & Fantasy,listed_in_TV Shows,listed_in_TV Thrillers,listed_in_Teen TV Shows,listed_in_Thrillers,duration_final
0,s1,Dick Johnson Is Dead,"As her father nears the end of his life, filmm...",0.989583,0.7,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.961405
1,s2,Blood & Water,"After crossing paths at a party, a Cape Town t...",1.000000,0.9,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.834492
2,s3,Ganglands,To protect his family from a powerful drug lor...,1.000000,0.9,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.890125
3,s4,Jailbirds New Orleans,"Feuds, flirtations and toilet talk go down amo...",1.000000,0.9,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.890125
4,s5,Kota Factory,In a city of coaching centers known to train I...,1.000000,0.9,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.834492


In [11]:
df_filtered.to_csv('netflix_encoded.csv', index=False, encoding='utf-8-sig')